In [1]:
import pandas  as pd
import numpy as np

In [21]:
# 提现用户及其子集
data1 = pd.read_csv('../临时文件/提现成功用户及其下级.csv',index_col= False)

In [22]:
data1 = data1[data1['son_Id']==data1['son_Id']]

In [23]:
data1['son_Id'] = data1['son_Id'].astype('int64')

In [31]:
# 用户获取代币数
data2 = pd.read_csv('../临时文件/用户获取代币数.csv',index_col=False)
data2.fillna(value=0,inplace=True)

In [32]:
data2['total_ipo']  = data2['ipo1']+data2['ipo2']

In [35]:
data2=data2.groupby(by='id1',as_index=False)['total_ipo'].sum()

In [82]:
data2['ipo_cut'] = pd.cut(data2['total_ipo'],
       bins=[100,200,2000,35000,10000000],# 分组的组数或者自定义分组列表
       right=False,# 是否包含右边的值
       labels=['1档','2档','3档','4档'],# 显示标签
       include_lowest=True # 是否包含最低值
                         ).astype('string')

In [83]:
data2.head()

,id1,total_ipo,ipo_cut
0,51646328,394.0,2档
1,58099130,1044.0,2档
2,98046501,114.0,1档
3,106309950,1028.0,2档
4,106445869,12524.0,3档


In [84]:
# 有效用户判定 
res = data1.merge(data2,how ='left',left_on='son_Id',right_on='id1')

In [87]:
res['ipo_cut'].value_counts()

ipo_cut
0档    97252
2档    11421
3档     3193
1档     2484
4档      332
Name: count, dtype: Int64

In [85]:
res['ipo_cut']=res['ipo_cut'].fillna(value='0档')

In [89]:
# 设置优化目标
# 有效用户数占比40% - 有效用户（用户总得分）/总用户 = 0.4

# 单用户最高分 2.5 分，最低分 0.5 + 
# 初始化得分 0 档位的用户0.1分
res['得分'] = res['ipo_cut'].apply(lambda x: 0.1 if x=='0档' else 0)
for i in range(5,26):
    res['得分'] = res.apply(lambda x: i/10 if x['ipo_cut']=='1档' else x['得分'],axis=1)
    for j in range(i+2,26):
        res['得分'] = res.apply(lambda x: j/10 if x['ipo_cut']=='2档' else x['得分'],axis=1)
        for k in range(j+4,26):
            res['得分'] = res.apply(lambda x: k/10 if x['ipo_cut']=='3档' else x['得分'],axis=1)
            for l in range(k+5,26):
                res['得分'] = res.apply(lambda x: l/10 if x['ipo_cut']=='4档' else x['得分'],axis=1)
                mark_sum = res['得分'].sum()
                users = res['son_Id'].nunique()
                if (mark_sum / users)>=0.4 and  (mark_sum / users) <=0.42 :
                    print(i,j,k,l)
                    break
                else:
                    pass
        
    

5 7 11 16
5 7 12 17
5 7 13 18


KeyboardInterrupt: 

In [73]:
dic1

{'0档': [0.1], '1档': [1], '2档': [1], '3档': [1], '4档': [1], '5档': [1]}